# Quickstart: Bose-Hubbard chain
This notebook is the example from the documentation start page.

In [1]:
import jax
from pygfmc import run
from pygfmc.walkers import prepare_walkers_sz
from pygfmc.hamiltonian import BoseHubbard
from pygfmc.wavefunction import JastrowBoseHubbard, train_with_netket
import os
os.environ["NETKET_NO_TIPS"] = "1"   # silence NetKet's tips
import netket as nk          # after pygfmc, so that everything stays in float32

print(f'JAX version : {jax.__version__}')
print(f'Backend     : {jax.default_backend()}')

JAX version : 0.9.2
Backend     : cpu


## Hamiltonian and walkers

We simulate the Bose-Hubbard model on a periodic chain with $L = 50$ sites and at most two bosons per site ($t = 1, \eta = 0.25, U = 4, \mu = 1$):

$$H = -\sum_i \left[ t \left( b^\dagger_i b_{i+1} + \mathrm{h.c.} \right) + \eta \left( b^\dagger_i b^\dagger_{i+1} + \mathrm{h.c.} \right) \right] + \frac{U}{2} \sum_i n_i (n_i - 1) - \mu \sum_i n_i\,.$$
The model conserves the particle number modulo two. All walkers are initialized in the even particle-number sector.

In [2]:
L       = 50
edges   = [[i, (i + 1) % L] for i in range(L)]   # periodic chain

H       = BoseHubbard.create(edges, t=1.0, eta=0.25, U=4.0, mu=1.0, n_max=2)
walkers = prepare_walkers_sz(512, L, jax.random.PRNGKey(0), sz=L, states=(0.0, 1.0, 2.0))

## Guiding wavefunction

The guide is a Gutzwiller-Jastrow state,

$$\log\psi_T(x) = -\beta\sum_i x_i x_{i+1} + h\sum_i x_i - d\sum_i x_i^2\,,$$

where $\beta$ correlates neighboring occupations, $h$ sets the density and $d$ suppresses doubly occupied sites. NetKet optimizes its three parameters:

In [3]:
guide   = JastrowBoseHubbard(L, edges, beta=0.0, h=0.0, d=0.5)
sampler = nk.sampler.MetropolisLocal(nk.hilbert.Fock(n_max=2, N=L))   # pair term: N not conserved
guide, history = train_with_netket(H, guide, sampler=sampler, n_iter=50, learning_rate=0.05,
                                   sr=True, qgt=nk.optimizer.qgt.QGTJacobianDense, return_history=True)
print(f"optimal parameters: beta = {float(guide.beta[0]):.3f}, h = {float(guide.h[0]):.3f}, d = {float(guide.d[0]):.3f}")
print(f"variational energy per site: {history[-1] / L:.5f}")

optimal parameters: beta = 0.246, h = 1.812, d = 0.680
variational energy per site: -2.01891


## Run

We run the continuous-time propagation with imaginary time `step_size = 0.25` per branching
step. We use a short warm-up of $10^3$ steps. The projection windows are `p = [8, 16, 32, 64]`.

The DMRG reference energy (TeNPy, bond dimension 384) is $E_{\mathrm{DMRG}} = -101.3171536$, or $E_{\mathrm{DMRG}}/L = -2.0263431$ per site.

In [4]:
walkers, energy, obs, stats, raw = run(
    walkers, n_steps=2000, hamiltonian=H, step_size=0.25,
    guide=guide, p_array=[8, 16, 32, 64], rng_key=jax.random.PRNGKey(1),
    warmup_steps=1000, verbosity=10
)

print({p: e / L for p, e in energy.items()})    # energy per site
print("Reference from DMRG for L=50: -2.026343072921")

[gfmc] - [run_warmup 1/4] - step=0.03125, 250 steps @ 18.7 it/s, avg_log_weight = 101.436760


[gfmc] - [run_warmup 2/4] - step=0.0625, 250 steps @ 14.5 it/s, avg_log_weight = 101.309677


[gfmc] - [run_warmup 3/4] - step=0.125, 250 steps @ 9.9 it/s, avg_log_weight = 101.314621


[gfmc] - [run_warmup 4/4] - step=0.25, 250 steps @ 6.2 it/s, avg_log_weight = 101.325027


[gfmc] - [run_gfmc] - step: 80/2000, 4.6 it/s, N_config = 450/512 (87.9%) Energy[p]: {8: -101.31175338330249, 16: -101.31410106026344, 32: -101.30557638686783, 64: -101.2906892279989}


...

[gfmc] - [run_gfmc] - step: 1940/2000, 4.6 it/s, N_config = 447/512 (87.3%) Energy[p]: {8: -101.31574554311318, 16: -101.31587212588296, 32: -101.31548512597487, 64: -101.31544913688857}


[gfmc] - [run_gfmc] - step: 1960/2000, 5.0 it/s, N_config = 451/512 (88.1%) Energy[p]: {8: -101.3156744073981, 16: -101.3157679490077, 32: -101.3154429065067, 64: -101.31534464580844}


[gfmc] - [run_gfmc] - step: 1980/2000, 5.5 it/s, N_config = 454/512 (88.7%) Energy[p]: {8: -101.31577995313428, 16: -101.3158623491317, 32: -101.31554230793591, 64: -101.31544606842259}


[gfmc] - [run_gfmc] - step: 2000/2000, 5.0 it/s, N_config = 443/512 (86.5%) Energy[p]: {8: -101.3156633019376, 16: -101.3157418733054, 32: -101.31542623270154, 64: -101.31534114521968}


[estimators] - [statistics] - 10 branching steps (+22 lineage-only)
[estimators] - [statistics] - N_walkers: 512
[estimators] - [statistics] - step_size: 0.25
[estimators] - [statistics] - lam: not used
[estimators] - [statistics] - avg_log_weight: 101.325027
[estimators] - [statistics] - unique (pre): 445.2 ± 3.8  (87.0%)
[estimators] - [statistics] - unique (post): 445.7 ± 3.8  (87.1%)
[estimators] - [statistics] - H_diag: min=-40.0000, max=12.0000  (observed max over the 10 health steps)
[estimators] - [statistics] - escape rate: min=60.7937, max=114.9707
[estimators] - [statistics] - hops per branching step (Gamma*step_size): mean=21.466, max=28.743
[estimators] - [statistics] - escape rate Gamma: mean=85.864, min=60.794, max=114.971
[estimators] - [statistics] - hold time (a draw of 1/Gamma, not hops per step): mean=0.0116, max=0.1046
[estimators] - [statistics] - changed (per-index): 100.0% avg over 10 steps
[estimators] - [statistics] - overlap (pop):       0.0% avg over 10 step